In [1]:
import warnings
warnings.filterwarnings("ignore")

## Imports

In [2]:
import math
import os
import time
import requests
import pandas as pd
import json
import numpy as np
from pathlib import Path
from typing import Optional, Dict, List, Union
from unidecode import unidecode
import matplotlib.pyplot as plt

import pandas_gbq
from google.auth import default
from google.cloud import bigquery
from google.api_core.exceptions import NotFound

In [3]:
from funcoes_escoragem import *

## Diretório

In [4]:
BASE_DIR = Path("data")
RAW_DIR = BASE_DIR / "raw"
TRUSTED_DIR = BASE_DIR / "trusted"
ANALYTICS_DIR = BASE_DIR / "analytics"

for path in [RAW_DIR, TRUSTED_DIR, ANALYTICS_DIR]:
    path.mkdir(parents=True, exist_ok=True)

## Query Performance

In [5]:
project_id = 'loft-dl-fintech'

query = '''
WITH
first_defaults AS (
  SELECT
    contract_id,
    DATE(MIN(pendency_created_at)) AS first_comunicacao_date,
    DATE(MIN(pendency_at)) AS first_competencia_date,
    DATE(MIN(payment_at)) AS first_payment_date
  FROM `loft-dl-fintech.cp_gold.watchlist_fact`
  WHERE pendency_type IN ('Inadimplência')
  GROUP BY contract_id
),

tb_base AS (
  SELECT
    cf.contract_id,
    DATE(rf.dt_lead) AS dt_lead,
    DATE(cf.requested_at) AS requested_at,
    DATE(rf.dt_proposta_iniciada) AS iniciada_at,
    DATE(rf.dt_proposta_enviada) AS enviada_at,
    DATE(rf.activated_at) AS activated_at,
    DATE(rf.cancelled_at) AS cancelled_at,
    DATE(rf.dt_saida) AS dt_saida,
    DATE_TRUNC(DATE(cf.requested_at), MONTH) AS safra,
    DATE_TRUNC(DATE(rf.activated_at), MONTH) AS safra_ativacao,

    cf.tipo_contrato,
    cf.tipo,
    rd.product_nm,
    cf.qtd_proponentes,
    CASE WHEN COALESCE(cf.qtd_proponentes, 0) > 1 THEN 1 ELSE 0 END AS is_multiproponente,

    cf.person_restriction_quantity,
    cf.person_restriction_total_value,
    cf.bureau_nm,
    cf.modeloBlend,
    cf.modelo_blend,
    cf.rating_score_ds,
    cf.score_imobiliaria,

    rd.agency_id,
    ad.segmentacao AS agency_segmentacao,
    cf.contract_city_nm,
    cf.person_age,
    cf.score_BVS_CUSTOM,
    cf.person_deemed_income,

    cf.blend_regressao_predict_nr,
    cf.bvs_cust_score_nr,
    cf.blendRegressaoPredict,
    cf.score_BVS_CUSTOM,

    CAST(rf.total_rental_value_informed_nr AS FLOAT64) AS total_rental_value_informed_nr,
    CAST(rf.rental_value_nr AS FLOAT64) AS rental_value_nr,

    rd.pre_analysis_result,
    rd.lead_elegivel,
    CASE WHEN CAST(cf.lead_elegivel AS STRING) = 'true' THEN 1 ELSE 0 END AS is_lead_elegivel,
    rd.proposta_iniciada,
    rd.proposta_enviada,
    rd.proposta_aprovada,
    rd.proposta_ativada,
    COALESCE(rd.is_activeted, cf.is_activeted) AS is_activeted,

    DATE_DIFF(
      DATE_TRUNC(DATE(CURRENT_DATE()), MONTH),
      DATE_TRUNC(COALESCE(DATE(rf.activated_at), DATE(cf.requested_at)), MONTH),
      MONTH
    ) AS time2requested,

    CASE
      WHEN fd.first_competencia_date IS NULL OR rf.activated_at IS NULL THEN NULL
      ELSE DATE_DIFF(
        DATE_TRUNC(DATE(fd.first_competencia_date), MONTH),
        DATE_TRUNC(DATE(rf.activated_at), MONTH),
        MONTH
      )
    END AS time2def_pc,

    CASE
      WHEN fd.first_comunicacao_date IS NULL OR rf.activated_at IS NULL THEN NULL
      ELSE DATE_DIFF(
        DATE_TRUNC(DATE(fd.first_comunicacao_date), MONTH),
        DATE_TRUNC(DATE(rf.activated_at), MONTH),
        MONTH
      )
    END AS time2def_pcc,

    CASE
      WHEN fd.first_payment_date IS NULL OR rf.activated_at IS NULL THEN NULL
      ELSE DATE_DIFF(
        DATE_TRUNC(DATE(fd.first_payment_date), MONTH),
        DATE_TRUNC(DATE(rf.activated_at), MONTH),
        MONTH
      )
    END AS time2def_pd,

    fd.first_comunicacao_date,
    fd.first_competencia_date,
    fd.first_payment_date
  FROM `loft-dl-fintech.cp_gold.credit_fact` AS cf
  LEFT JOIN `loft-dl-fintech.cp_gold.requests_fact` AS rf
    ON cf.contract_id = rf.contract_id
  LEFT JOIN `loft-dl-fintech.cp_gold.requests_dim` AS rd
    ON cf.contract_id = rd.contract_id
  LEFT JOIN `loft-dl-fintech.cp_gold.agency_dim` AS ad
    ON rd.agency_id = ad.agency_id
  LEFT JOIN first_defaults AS fd
    ON cf.contract_id = fd.contract_id
  WHERE DATE(cf.requested_at) >= DATE('2024-01-01') AND rf.activated_at IS NOT NULL
),

tb_flags AS (
  SELECT
  *,
    CASE
      WHEN activated_at IS NULL THEN NULL
      WHEN first_competencia_date IS NOT NULL AND time2requested >= 4 AND time2def_pc <= 3 THEN 1
      WHEN time2requested >= 4 THEN 0
      ELSE NULL
    END AS pc_2m,
    CASE
      WHEN activated_at IS NULL THEN NULL
      WHEN first_competencia_date IS NOT NULL AND time2requested >= 5 AND time2def_pc <= 3 THEN 1
      WHEN time2requested >= 5 THEN 0
      ELSE NULL
    END AS pc_3m,
    CASE
      WHEN activated_at IS NULL THEN NULL
      WHEN first_competencia_date IS NOT NULL AND time2requested >= 8 AND time2def_pc <= 6 THEN 1
      WHEN time2requested >= 8 THEN 0
      ELSE NULL
    END AS pc_6m,
    CASE
      WHEN activated_at IS NULL THEN NULL
      WHEN first_competencia_date IS NOT NULL AND time2requested >= 11 AND time2def_pc <= 9 THEN 1
      WHEN time2requested >= 11 THEN 0
      ELSE NULL
    END AS pc_9m,
    CASE
      WHEN activated_at IS NULL THEN NULL
      WHEN first_competencia_date IS NOT NULL AND time2requested >= 14 AND time2def_pc <= 12 THEN 1
      WHEN time2requested >= 14 THEN 0
      ELSE NULL
    END AS pc_12m,

    CASE
      WHEN activated_at IS NULL THEN NULL
      WHEN first_payment_date IS NOT NULL AND time2requested >= 2 AND time2def_pd <= 3 THEN 1
      WHEN time2requested >= 2 THEN 0
      ELSE NULL
    END AS pd_2m,
    CASE
      WHEN activated_at IS NULL THEN NULL
      WHEN first_payment_date IS NOT NULL AND time2requested >= 3 AND time2def_pd <= 3 THEN 1
      WHEN time2requested >= 3 THEN 0
      ELSE NULL
    END AS pd_3m,
    CASE
      WHEN activated_at IS NULL THEN NULL
      WHEN first_payment_date IS NOT NULL AND time2requested >= 6 AND time2def_pd <= 6 THEN 1
      WHEN time2requested >= 6 THEN 0
      ELSE NULL
    END AS pd_6m,
    CASE
      WHEN activated_at IS NULL THEN NULL
      WHEN first_payment_date IS NOT NULL AND time2requested >= 9 AND time2def_pd <= 9 THEN 1
      WHEN time2requested >= 9 THEN 0
      ELSE NULL
    END AS pd_9m,
    CASE
      WHEN activated_at IS NULL THEN NULL
      WHEN first_payment_date IS NOT NULL AND time2requested >= 12 AND time2def_pd <= 12 THEN 1
      WHEN time2requested >= 12 THEN 0
      ELSE NULL
    END AS pd_12m,

    CASE
      WHEN activated_at IS NULL THEN NULL
      WHEN first_comunicacao_date IS NOT NULL AND time2requested >= 2 AND time2def_pcc <= 3 THEN 1
      WHEN time2requested >= 2 THEN 0
      ELSE NULL
    END AS pcc_2m,
    CASE
      WHEN activated_at IS NULL THEN NULL
      WHEN first_comunicacao_date IS NOT NULL AND time2requested >= 3 AND time2def_pcc <= 3 THEN 1
      WHEN time2requested >= 3 THEN 0
      ELSE NULL
    END AS pcc_3m,
    CASE
      WHEN activated_at IS NULL THEN NULL
      WHEN first_comunicacao_date IS NOT NULL AND time2requested >= 6 AND time2def_pcc <= 6 THEN 1
      WHEN time2requested >= 6 THEN 0
      ELSE NULL
    END AS pcc_6m,
    CASE
      WHEN activated_at IS NULL THEN NULL
      WHEN first_comunicacao_date IS NOT NULL AND time2requested >= 9 AND time2def_pcc <= 9 THEN 1
      WHEN time2requested >= 9 THEN 0
      ELSE NULL
    END AS pcc_9m,
    CASE
      WHEN activated_at IS NULL THEN NULL
      WHEN first_comunicacao_date IS NOT NULL AND time2requested >= 12 AND time2def_pcc <= 12 THEN 1
      WHEN time2requested >= 12 THEN 0
      ELSE NULL
    END AS pcc_12m
  FROM tb_base
)

SELECT *
FROM tb_flags
'''

In [6]:
# df_sim_ctr = pd.read_gbq(query, project_id=project_id)
# df_sim_ctr.to_parquet(f"{BASE_DIR}/base_sim_ctr_performance.parquet")

In [7]:
df_sim_ctr = pd.read_gbq(query, project_id=project_id)
df_sim_ctr.to_parquet(f"{BASE_DIR}/base_ctr_performance.parquet")

In [8]:
df_sim_ctr

,contract_id,dt_lead,requested_at,iniciada_at,enviada_at,activated_at,cancelled_at,dt_saida,safra,safra_ativacao,...,pd_2m,pd_3m,pd_6m,pd_9m,pd_12m,pcc_2m,pcc_3m,pcc_6m,pcc_9m,pcc_12m
0,1993142,2024-10-18,2024-10-18,2024-10-18,2024-10-18,2020-10-18,NaT,NaT,2024-10-01,2020-10-01,...,0,0,0,0,0,0,0,0,0,0
1,1494718,2024-01-02,2024-01-02,NaT,2024-01-02,2024-01-05,2025-11-26,2025-11-19,2024-01-01,2024-01-01,...,0,0,0,0,0,0,0,0,0,0
2,1494591,2024-01-02,2024-01-02,NaT,2024-01-02,2024-01-03,NaT,2025-09-30,2024-01-01,2024-01-01,...,0,0,1,1,1,1,1,1,1,1
3,1495076,2024-01-02,2024-01-02,NaT,2024-01-02,2024-01-03,NaT,NaT,2024-01-01,2024-01-01,...,0,0,0,0,0,0,0,1,1,1
4,1495205,2024-01-02,2024-01-02,NaT,2024-01-02,2024-01-02,NaT,NaT,2024-01-01,2024-01-01,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
513620,4459748,2026-08-04,2026-08-04,2026-08-04,2026-08-04,2026-08-04,NaT,NaT,2026-08-01,2026-08-01,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
513621,4462026,2026-08-04,2026-08-04,2026-08-04,2026-08-04,2026-08-05,NaT,NaT,2026-08-01,2026-08-01,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
513622,4452266,2026-08-01,2026-08-01,2026-08-04,2026-08-04,2026-08-04,NaT,NaT,2026-08-01,2026-08-01,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
513623,4451458,2026-08-01,2026-08-01,2026-08-04,2026-08-04,2026-08-04,NaT,NaT,2026-08-01,2026-08-01,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
